# Import libraries

In [ ]:
from experiment_common_code import ExperimentResult, plot_metrics_by_group, plot_confusion_matrix, show_plots, ExperimentResult
from utils import LANGUAGES, MODEL_NAMES, MODEL_THESIS_NAMES, REVERSE_LABEL_MAP, PROBE_TYPE_FULL_NAME_MAP, get_language_pair_permutations, get_all_language_merged_strings
from icecream import ic
import pandas as pd
from pandas import DataFrame

import torch as t

from typing import Literal, Any

device: Literal["cuda", "cpu"] = "cuda" if t.cuda.is_available() else "cpu"

# Experiment 1

Load important variables

In [ ]:
from experiment_1 import run_experiment_1

model_names: list[str] = MODEL_NAMES
languages: list[str] = LANGUAGES
probe_type: str = "lr"
num_layers: int | None = None

custom = False
if custom:
    model_names = ["olmo_model"]
    languages = ["en"]
    print(f"Using custom configuration")

ic(custom, model_names, languages, probe_type)

Run experiment 1

In [ ]:
limited_layers = False
if limited_layers:
    num_layers = 3

force_probe_creation: bool = False

save_results: bool = True

ic(limited_layers, force_probe_creation, save_results)

run_experiment_1(languages, "standard", "control", probe_type, model_names, force_probe_creation, num_layers=num_layers, save_results=save_results)

In [ ]:
example_exp_result: ExperimentResult = ExperimentResult.get_from_file(
                        1, "en", "standard", probe_type, "olmo_model"
                    )
print(example_exp_result.metrics["test"].keys())
for layer_num in range(example_exp_result.get_num_layers()):
    print(f"Layer {layer_num}")
    # overlapping_idxs: dict[str, set[int]] = example_exp_result.overlapping_idxs["test"]
    overlapping_idxs: dict[str, set[int]] = example_exp_result.get_metric("test", "cummulative_overlapping_idxs", layer_num)
    for key, value in overlapping_idxs.items():
        real_value = key[5]
        prediciton = key[-1]
        if real_value != prediciton:
            # print(key)
            # print(value)
            print(f"{key}→{len(value)}")
    
    

Make some plots

In [ ]:
# Get list of plots showing some arbitrary metric
metric = "f1"
# metric = "per_class_precision"

show_plots(1, 
           model_names, 
           ["test"], 
           languages, 
           ["standard", "control"], 
        #    zeroed_out_weight_dims_list=[0, 5],
           metric=metric,
           legend_position="lower right",
           probe_type=probe_type,
           )

In [ ]:
metric = "previous_layer_overlapping_idx_amounts"

show_plots(1, 
           model_names, 
           ["test"], 
           languages, 
           ["standard"], 
           metric=metric,
           separate_chars_within_plot=["class_name"],
           legend_position="upper right"
           )

In [ ]:
# Get list of plots showing f1 for just the test set
metric = "f1"
show_plots(1, 
           model_names, 
           ["test"], 
           languages, 
           ["standard", "control"], 
           metric=metric)

In [ ]:
# for model_name in model_names:
#     for split in ["test"]:
#         for language in languages:
#             for probing_task in probing_tasks:
#                 exp_result: ExperimentResult = ExperimentResult.get_from_file(
#                     1, language, probing_task, probe_type, model_name
#                 )

#                 for layer_num in range(exp_result.get_num_layers()):
#                     plot_confusion_matrix(exp_result, split, layer_num)

for probing_task in ["standard", "control"]:
    exp_result: ExperimentResult = ExperimentResult.get_from_file(1, "en", probing_task, probe_type, "olmo_model")

    for layer_num in range(exp_result.get_num_layers()):
        plot_confusion_matrix(exp_result, "test", layer_num)

# Experiment 2

Load important variables

In [ ]:
from experiment_2 import run_experiment_2

model_names: list[str] = MODEL_NAMES
language_pairs: list[tuple[str, str]] = get_language_pair_permutations(LANGUAGES)
probe_type: str = "lr"

custom = False
if custom:
    model_names = ["olmo_model"]
    language_pairs = [("en", "en"), ("es", "es"), ("jp", "jp")]
    num_refits = 2
    print(f"Using custom configuration")

language_pairs_as_strings: list[str] = get_all_language_merged_strings(language_pairs)

ic(custom, model_names, language_pairs, language_pairs_as_strings, probe_type)

Run experiment 2

In [ ]:
num_layers: int | None = None

limited_layers = False
if limited_layers:
    num_layers = 5

force_probe_creation: bool = False
force_refit_probe_creation: bool = True

save_results: bool = True

ic(limited_layers, num_layers, force_probe_creation, force_refit_probe_creation, save_results)

run_experiment_2(language_pairs, 
                 "standard", 
                 "control", 
                 probe_type, 
                 model_names, 
                 num_refits=num_refits,
                 iterations_per_refit=iterations_per_refit,
                 force_probe_creation=force_probe_creation, 
                 force_refit_probe_creation=force_refit_probe_creation,
                 num_layers=num_layers, 
                 save_results=save_results,
                 )

Make some plots

In [ ]:
extra_iter_nums = [0, 1000, 2000]
show_plots(2, 
           model_names, 
           ["test_a", "test_b"],
           language_pairs_as_strings, 
           ["standard"], 
           list(extra_iter_nums), 
           metric="f1",
           separate_chars_within_plot=["split"],
           probe_type=probe_type
           )

In [ ]:
language_pairs_as_strings_to_show: list[str] = language_pairs_as_strings
# language_pairs_as_strings_to_show = ["en→es"]

extra_iter_num_type = 1

match extra_iter_num_type:
    case 0:
        extra_iter_nums: list[int] = list(range(0, 5, 1))
    case 1:
        extra_iter_nums = [0, 1, 2]
    case 2:
        extra_iter_nums = [0, 1000, 2000]
    case 3:
        extra_iter_nums = [0]

for extra_iter_num in extra_iter_nums:
    show_plots(2, 
            ["olmo_model"], 
            ["test_a", "test_b"],
            language_pairs_as_strings_to_show,
            ["standard", "control"], 
            [extra_iter_num],
            metric="f1",
            separate_chars_within_plot=["split", "probing_task"],
            save=True,
            filename=f"experiment_2_f1_per_layers_olmo_{extra_iter_num}_extra_iters",
            show=False,
            probe_type=probe_type,
            )
    show_plots(2, 
            ["tiny_aya_global"], 
            ["test_a", "test_b"],
            language_pairs_as_strings_to_show, 
            ["standard", "control"], 
            [extra_iter_num], 
            metric="f1",
            separate_chars_within_plot=["split", "probing_task"],
            save=True,
            filename=f"experiment_2_f1_per_layers_aya_{extra_iter_num}_extra_iters",
            show=False,
            probe_type=probe_type,
            )

In [ ]:
language_pairs_as_strings_to_show: list[str] = language_pairs_as_strings
# language_pairs_as_strings_to_show = ["en→es"]

extra_iter_num_type = 1

match extra_iter_num_type:
    case 0:
        extra_iter_nums: list[int] = list(range(0, 5, 1))
    case 1:
        extra_iter_nums = [0, 1, 2]
    case 2:
        extra_iter_nums = [0, 1000, 2000]
    case 3:
        extra_iter_nums = [0]

for extra_iter_num in extra_iter_nums:
    show_plots(2, 
            ["olmo_model"], 
            ["test_b"],
            language_pairs_as_strings_to_show,
            ["standard", "control"], 
            [extra_iter_num],
            metric="f1",
            separate_chars_within_plot=["language_b", "probing_task"],
            save=False,
            # filename=f"experiment_2_f1_per_layers_olmo_{extra_iter_num}_extra_iters",
            show=True,
            probe_type=probe_type,
            # zeroed_out_weight_dims_list=[0, 5, 100],
            legend_position="lower right",
            )
    show_plots(2, 
            ["tiny_aya_global"], 
            ["test_b"],
            language_pairs_as_strings_to_show, 
            ["standard", "control"], 
            [extra_iter_num], 
            metric="f1",
            separate_chars_within_plot=["language_b", "probing_task"],
            save=False,
            # filename=f"experiment_2_f1_per_layers_aya_{extra_iter_num}_extra_iters",
            show=True,
            probe_type=probe_type,
            # zeroed_out_weight_dims_list=[0, 5, 100],
            legend_position="lower right",
            )

In [ ]:
extra_iter_nums = [2000]

show_plots(2, 
        ["olmo_model"], 
        ["test_a", "test_b"],
        language_pairs_as_strings_to_show,
        ["standard"], 
        extra_iter_nums,
        metric="f1",
        separate_chars_within_plot=["language_b", "split"],
        save=False,
        # filename=f"experiment_2_f1_per_layers_olmo_{extra_iter_num}_extra_iters",
        show=True,
        probe_type=probe_type,
        # zeroed_out_weight_dims_list=[0, 5, 100],
        legend_position="lower right",
        )

show_plots(2, 
        ["tiny_aya_global"], 
        ["test_a", "test_b"],
        language_pairs_as_strings_to_show,
        ["standard"], 
        extra_iter_nums,
        metric="f1",
        separate_chars_within_plot=["language_b", "split"],
        save=False,
        # filename=f"experiment_2_f1_per_layers_olmo_{extra_iter_num}_extra_iters",
        show=True,
        probe_type=probe_type,
        # zeroed_out_weight_dims_list=[0, 5, 100],
        legend_position="lower right",
        )

# Experiment 3

Load important variables

In [ ]:
from experiment_3 import run_experiment_3, E3DataframeCreator

model_names: list[str] = MODEL_NAMES
languages: list[str] = LANGUAGES

custom = False
if custom:
    model_names = ["olmo_model"]
    languages = ["en"]
    print(f"Using custom configuration")

ic(custom, model_names, languages)

In [ ]:
run_experiment_3(languages, model_names)
# run_experiment_3(["nl"], ["olmo_model"])
# run_experiment_3(["es", "nl"], ["tiny_aya_global"])

In [ ]:
dataframe_creator_olmo = E3DataframeCreator("olmo_model", languages, include_strict=False)
df_e3_olmo: DataFrame = dataframe_creator_olmo.create_dataframe()
print(df_e3_olmo.to_latex(float_format="%.2f"))
df_e3_olmo

In [ ]:
dataframe_creator_aya = E3DataframeCreator("tiny_aya_global", languages, include_strict=False)
df_e3_aya: DataFrame = dataframe_creator_aya.create_dataframe()
print(df_e3_aya.to_latex(float_format="%.2f"))
df_e3_aya

In [ ]:
for model_name in model_names:
    for language in languages:
        exp_result: ExperimentResult = ExperimentResult.get_from_file(3, language, "standard", "model_pred", model_name)
        plot_confusion_matrix(exp_result, "test", include_unknown=True)

# Final paper plots

In [ ]:
for probe_type in ["lr", "mm"]:
        show_plots(1, 
                MODEL_NAMES, 
                ["test"], 
                LANGUAGES, 
                ["standard"], 
                metric="f1",
                probe_type=probe_type,
                legend_position="upper left",
                subplot_titles=[f"{MODEL_THESIS_NAMES[model_name]}".capitalize() for model_name in MODEL_NAMES],
                y_axis_range=(0.45, 1.0),
                show=True,
                save=True,
                filename=f"e1_f1_{probe_type}.png",
                as_bars=False,
                )

In [ ]:
for probe_type in ["lr", "mm"]:
        show_plots(1, 
                MODEL_NAMES, 
                ["test"], 
                LANGUAGES, 
                ["standard"], 
                metric="f1",
                probe_type=probe_type,
                legend_position="upper left",
                subplot_titles=[f"{MODEL_THESIS_NAMES[model_name]}".capitalize() for model_name in MODEL_NAMES],
                y_axis_range=(0.45, 1.0),
                show=False,
                as_bars=False,
                print_lines=True,
                layer_to_print=-1
                )

In [ ]:
for probe_type in ["lr", "mm"]:
    show_plots(1, 
                    MODEL_NAMES, 
                    ["test"], 
                    LANGUAGES, 
                    ["standard"], 
                    metric="marginal_f1",
                    probe_type=probe_type,
                    horizontal_line=0,
                    legend_position="upper left",
                    subplot_titles=[f"{MODEL_THESIS_NAMES[model_name]}".capitalize() for model_name in MODEL_NAMES],
                    y_axis_range=(-0.3, 0.4),
                    show=False,
                    as_bars=False,
                    print_lines=True
                    )

In [ ]:
for probe_type in ["lr", "mm"]:
        show_plots(1, 
                MODEL_NAMES, 
                ["test"], 
                LANGUAGES, 
                ["standard"], 
                metric="marginal_f1",
                probe_type=probe_type,
                horizontal_line=0,
                legend_position="upper left",
                subplot_titles=[f"{MODEL_THESIS_NAMES[model_name]}".capitalize() for model_name in MODEL_NAMES],
                y_axis_range=(-0.3, 0.4),
                show=True,
                save=True,
                filename=f"e1_selectivity_{probe_type}.png",
                as_bars=False,
                )

In [ ]:
# for probe_type in ["lr", "mm"]:
#         show_plots(1, 
#                 MODEL_NAMES, 
#                 ["test"], 
#                 LANGUAGES, 
#                 ["standard"], 
#                 metric="marginal_f1",
#                 probe_type=probe_type,
#                 horizontal_line=0,
#                 legend_position="upper left",
#                 subplot_titles=[f"{MODEL_THESIS_NAMES[model_name]}".capitalize() for model_name in MODEL_NAMES],
#                 y_axis_range=(-0.3, 0.3),
#                 show=True,
#                 # save=True,
#                 # filename=f"e1_selectivity_{probe_type}.png",
#                 as_bars=False,
#                 print_lines=True,
#                 # layer_to_print=-1
#                 )

In [ ]:
def make_experiment_2_plot(model_name, probe_type, extra_iter_num, legend_position="upper left", language_pairs: list[tuple[str, str]]|None=None, filename_suffix="", as_bars=True) -> None:
    filename = f"e2_{probe_type}_{extra_iter_num}_{model_name}{'_ablation' if language_pairs is not None else ''}{'_'+filename_suffix if filename_suffix else ''}"

    if language_pairs is None:
        language_pairs = get_language_pair_permutations(LANGUAGES)
    
    print(language_pairs)
    
    language_pairs_as_strings: list[str] = get_all_language_merged_strings(language_pairs)
    
    print(f"Creating plot of {model_name} {probe_type} with {extra_iter_num} extra iters")
    show_plots(2, 
                [model_name], 
                ["test_b", "test_a"],
                language_pairs_as_strings,
                ["standard"], 
                [extra_iter_num],
                metric="f1",
                separate_chars_within_plot=["language_b", "split", "probing_task"],
                probe_type=probe_type,
                horizontal_line="baseline_f1",
                legend_position=legend_position,
                subplot_titles=[],
                y_axis_range=(0.0, 1.0),
                save=True,
                show=True,
                filename=filename,
                as_bars=as_bars
                )

In [ ]:
# 0 extra iters
for probe_type in ["lr", "mm"]:
    for model_name in MODEL_NAMES:
        make_experiment_2_plot(model_name, probe_type, 0)

In [ ]:
# #  1 extra iter
# for model_name in model_names:
#     make_experiment_2_plot(model_name, "lr", 1, legend_position="lower center")

In [ ]:
#  2 extra iters
for model_name in MODEL_NAMES:
    make_experiment_2_plot(model_name, "lr", 2, legend_position="lower center")

In [ ]:
# #  10 extra iters
# for model_name in MODEL_NAMES:
#     make_experiment_2_plot(model_name, "lr", 10, legend_position="lower center")

In [ ]:
#  1000 extra iters
for model_name in MODEL_NAMES:
    make_experiment_2_plot(model_name, "lr", 1000, legend_position="lower center")

In [ ]:
# Ablation: Japanese with original labels
probe_type = "lr"
# for probe_type in ["lr", "mm"]:
show_plots(1, 
        MODEL_NAMES, 
        ["test"], 
        ["jp", "jp_original_labels"], 
        ["standard"], 
        metric="f1",
        probe_type=probe_type,
        legend_position="upper left",
        subplot_titles=[f"{MODEL_THESIS_NAMES[model_name]}".capitalize() for model_name in MODEL_NAMES],
        y_axis_range=(0.3, 1.0),
        show=True,
        save=True,
        filename=f"e1_f1_{probe_type}_jp_ablation.png",
        as_bars=False,
        )

show_plots(1, 
        MODEL_NAMES, 
        ["test"], 
        ["jp", "jp_original_labels"], 
        ["standard"], 
        metric="marginal_f1",
        probe_type=probe_type,
        horizontal_line=0,
        legend_position="upper left",
        subplot_titles=[f"{MODEL_THESIS_NAMES[model_name]}".capitalize() for model_name in MODEL_NAMES],
        y_axis_range=(-0.3, 0.6),
        show=True,
        save=True,
        filename=f"e1_selectivity_{probe_type}_jp_ablation.png",
        as_bars=False,
        )

In [ ]:
probe_type="lr"
# for probe_type in ["lr", "mm"]:
for model_name in MODEL_NAMES:
    make_experiment_2_plot(model_name, probe_type, 0, language_pairs=[
        ("en", "jp"), ("es", "jp"), ("nl", "jp"),
        ("en", "jp_original_labels"), ("es", "jp_original_labels"), ("nl", "jp_original_labels"),
    ],filename_suffix="1")

    make_experiment_2_plot(model_name, probe_type, 0, language_pairs=[
        ("jp", "en"), ("jp", "es"), ("jp", "nl"),
        ("jp_original_labels", "en"), ("jp_original_labels", "es"), ("jp_original_labels", "nl"),
    ],filename_suffix="2")

In [ ]:
# line plot ablations
for model_name in MODEL_NAMES:
        make_experiment_2_plot(model_name, "lr", 0, as_bars=False)
        make_experiment_2_plot(model_name, "lr", 2, as_bars=False)
        make_experiment_2_plot(model_name, "lr", 1000, as_bars=False)